This notebook gives you:

- a claims severity classifier
- using Bio_ClinicalBERT
- with 3 severity levels
- train/val split, metrics, and inference

In [1]:
# GPU check
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


In [2]:
# imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

In [3]:
# load BIO_ClinicalBERT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


##### CHANGE 1: For large state of art model (64GB)
MODEL_NAME = "microsoft/deberta-v3-large"   # or longformer / bigbird / clinical-longformer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)

Transformers says:
- “I found the encoder weights”
- “But I didn’t find a classifier head” - that's what you want for fine-tuning
- training will optimise the classifier head

In [4]:
# synthetic claims dataset
data = {
    "claim_text": [
        "Patient presents with chest pain radiating to left arm and shortness of breath. ECG shows ST elevation. Admitted to cardiology.",
        "Post-operative patient with fever and elevated CRP. Wound site appears red and swollen. Treated with IV antibiotics.",
        "Elderly patient with history of COPD, increased breathlessness over 3 days, requiring nebulisers and oxygen.",
        "Patient attended A&E following a fall. X-ray confirms fractured wrist. Plaster applied, discharged with follow-up.",
        "Patient reports mild headache and fatigue. Observations within normal range. Discharged with advice to rest.",
        "Patient experiencing dizziness and blurred vision. Blood pressure elevated. Started on antihypertensive medication.",
        "Patient complains of sore throat and runny nose. No fever. Managed with symptomatic treatment.",
        "Patient with abdominal pain, nausea, and vomiting. CT scan suggests appendicitis. Taken to theatre for appendicectomy.",
        "Patient with minor laceration to forearm. Cleaned and dressed. No complications.",
        "Patient with sepsis secondary to pneumonia. Admitted to ICU for vasopressor support."
    ],
    "severity_label": [
        2,  # MI suspicion, admission
        2,  # post-op infection, IV antibiotics
        2,  # COPD exacerbation, oxygen
        1,  # fracture, treated, follow-up
        0,  # mild headache
        1,  # hypertensive episode, meds
        0,  # mild URTI
        2,  # appendicitis, surgery
        0,  # minor laceration
        2   # sepsis, ICU
    ]
}

df = pd.DataFrame(data)
df

,claim_text,severity_label
0,Patient presents with chest pain radiating to ...,2
1,Post-operative patient with fever and elevated...,2
2,"Elderly patient with history of COPD, increase...",2
3,Patient attended A&E following a fall. X-ray c...,1
4,Patient reports mild headache and fatigue. Obs...,0
5,Patient experiencing dizziness and blurred vis...,1
6,Patient complains of sore throat and runny nos...,0
7,"Patient with abdominal pain, nausea, and vomit...",2
8,Patient with minor laceration to forearm. Clea...,0
9,Patient with sepsis secondary to pneumonia. Ad...,2


In [5]:
# train/validation split
train_df = df.sample(frac=0.7, random_state=42)
val_df = df.drop(train_df.index)

train_df, val_df

(                                          claim_text  severity_label
 8  Patient with minor laceration to forearm. Clea...               0
 1  Post-operative patient with fever and elevated...               2
 5  Patient experiencing dizziness and blurred vis...               1
 0  Patient presents with chest pain radiating to ...               2
 7  Patient with abdominal pain, nausea, and vomit...               2
 2  Elderly patient with history of COPD, increase...               2
 9  Patient with sepsis secondary to pneumonia. Ad...               2,
                                           claim_text  severity_label
 3  Patient attended A&E following a fall. X-ray c...               1
 4  Patient reports mild headache and fatigue. Obs...               0
 6  Patient complains of sore throat and runny nos...               0)

In [ ]:
# tokenisation
def tokenize_batch(batch):
    return tokenizer(
        batch["claim_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

##### CHANGE 2: For large state of art model (64GB) - increase context length

def tokenize_batch(batch):
    return tokenizer(
        batch["claim_text"],
        padding="max_length",
        truncation=True,
        max_length=512    # or 2048 for longformer/bigbird
    )

In [ ]:
# Tokenise
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)

# FIX: rename label column
train_ds = train_ds.rename_column("severity_label", "labels")
val_ds = val_ds.rename_column("severity_label", "labels")

# Remove only text + index
train_ds = train_ds.remove_columns(["claim_text", "__index_level_0__"])
val_ds = val_ds.remove_columns(["claim_text", "__index_level_0__"])

# Torch format
train_ds.set_format("torch")
val_ds.set_format("torch")

In [ ]:
# metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1_macro}

In [ ]:
# training arguments + trainer
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./claims-severity-model",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none"
)

NOTE:
With 64 GB VRAM, you can safely do:
- DeBERTa‑Large → batch 16
- Longformer‑Large → batch 8
- BigBird‑Pegasus‑Large → batch 4–8

##### CHANGE 3-5: For large state of art model (64GB)
training_args = TrainingArguments(
    output_dir="./clinicalbert-finetune",
    num_train_epochs=5,

    # CHANGE 3 — bigger batch size (because you have huge memory)
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",

    # CHANGE 4 — fp16 or bf16 mixed precision
    fp16=True,     # if your GPU supports FP16
    # bf16=True    # if your GPU supports BF16 (A100 / H100 / 4090)

    # CHANGE 5 — gradient checkpointing (critical for large models) - reduces memory by 30–40%.
    gradient_checkpointing=True
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [ ]:
# train
trainer.train()

In [ ]:
# evaluate
metrics = trainer.evaluate()
metrics

In [ ]:
# inference helper
SEVERITY_MAP = {
    0: "LOW",
    1: "MODERATE",
    2: "HIGH"
}

def predict_claim_severity(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    preds = np.argmax(probs, axis=-1)
    return preds, probs


In [ ]:
# test on new claims
test_claims = [
    "Patient presents to A&E with severe chest pain and hypotension. Transferred urgently to cath lab.",
    "Patient reports mild ankle sprain after walking. No fracture on X-ray. Discharged with advice.",
    "Post-operative patient with low-grade fever, responding to oral antibiotics.",
    "Patient with septic shock requiring vasopressors and mechanical ventilation."
]

preds, probs = predict_claim_severity(test_claims)

for text, label, p in zip(test_claims, preds, probs):
    print("\nCLAIM TEXT:\n", text)
    print("PREDICTED SEVERITY:", SEVERITY_MAP[int(label)])
    print("PROBS:", p)
